# 05 — Model Selection and Evidence Summary

This notebook consolidates the saved benchmark, control, and LOSO outputs for the selected dataset and produces a model-selection evidence summary.

## Purpose

The goal of this notebook is to decide which model should be carried forward as the main classification model for the thesis, using evidence from:

1. the **main within-subject grouped benchmark**
2. the **anti-overfitting controls**
   - permuted labels
   - pre-stimulus-only
3. the **cross-subject LOSO benchmark**

## Models considered

- **LDA**
- **Logistic Regression**
- **EEGNet**

## Selection philosophy

The final model should not be chosen using only the highest single score.
Instead, the selected model should satisfy all of the following as much as possible:

- strong performance on the main benchmark
- reasonable robustness across subjects
- acceptable seed stability
- evidence that performance is not driven by leakage or trivial structure
- reasonable behavior under cross-subject generalization
- practical suitability for the thesis

## Decision principle

This notebook treats the **within-subject grouped benchmark** as the primary model-selection benchmark.

The **controls** are used as validity checks.

The **LOSO benchmark** is used as supporting evidence about cross-subject generalization.

The final recommendation should therefore be:
- honest
- evidence-based
- and suitable for thesis reporting

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import pandas as pd

from oddball_eeg.config import get_experiment_family
from oddball_eeg.io import save_dataframe, save_experiment_metadata
from oddball_eeg.visualization import (
    make_main_vs_control_table,
    make_model_ranking_table,
    plot_main_vs_control,
    plot_model_ranking_bars,
    print_model_ranking,
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)

In [ ]:
set_pandas_display_options()

## 1. Load experiment directories

We load the output directories corresponding to:

- main within-subject benchmark
- permuted-label control
- pre-stimulus-only control
- LOSO benchmark

In [ ]:
# Change only this line when switching datasets.
# EXPERIMENT_NAME = "ds003061_within_subject_main"
# EXPERIMENT_NAME = "ds005863_within_subject_main"
EXPERIMENT_NAME = "ds006480_within_subject_main"

experiment_family = get_experiment_family(EXPERIMENT_NAME)

cfg_main = experiment_family["main"]
cfg_perm = experiment_family["permuted"]
cfg_prestim = experiment_family["prestim"]
cfg_loso = experiment_family["loso"]

MAIN_SUMMARY_DIR = cfg_main.table_dir("summaries")
PERM_SUMMARY_DIR = cfg_perm.table_dir("summaries")
PRESTIM_SUMMARY_DIR = cfg_prestim.table_dir("summaries")
LOSO_SUMMARY_DIR = cfg_loso.table_dir("summaries")

FINAL_TABLE_DIR = cfg_main.artifact_dir("model_selection", "tables")
FINAL_FIG_DIR = cfg_main.artifact_dir("model_selection", "figures")

save_experiment_metadata(
    cfg_main,
    extra={
        "notebook": "05_final_model_selection.ipynb",
        "purpose": "Consolidate benchmark, controls, and LOSO evidence for model-selection summary.",
        "experiment_key": EXPERIMENT_NAME,
        "main_summary_dir": str(MAIN_SUMMARY_DIR),
        "permuted_summary_dir": str(PERM_SUMMARY_DIR),
        "prestim_summary_dir": str(PRESTIM_SUMMARY_DIR),
        "loso_summary_dir": str(LOSO_SUMMARY_DIR),
        "final_table_dir": str(FINAL_TABLE_DIR),
        "final_figure_dir": str(FINAL_FIG_DIR),
    },
)

print("Experiment key:", EXPERIMENT_NAME)
print("Dataset       :", cfg_main.dataset.dataset)
print("Task          :", cfg_main.dataset.task)
print("Signal config :", cfg_main.signal)

print("\nInput summary directories:")
print("Main benchmark :", MAIN_SUMMARY_DIR)
print("Permuted       :", PERM_SUMMARY_DIR)
print("Pre-stimulus   :", PRESTIM_SUMMARY_DIR)
print("LOSO           :", LOSO_SUMMARY_DIR)

print("\nFinal artifact directories:")
print("Tables :", FINAL_TABLE_DIR)
print("Figures:", FINAL_FIG_DIR)

## 2. Load saved summary tables

This notebook assumes that the previous notebooks have already saved their outputs.

In [ ]:
# Main benchmark
df_main_subject_seed = pd.read_csv(MAIN_SUMMARY_DIR / "subject_seed_summary.csv")
df_main_subject = pd.read_csv(MAIN_SUMMARY_DIR / "subject_summary_across_seeds.csv")
df_main_model = pd.read_csv(MAIN_SUMMARY_DIR / "model_summary_across_subjects.csv")

# Permuted-label control
df_perm_subject_seed = pd.read_csv(PERM_SUMMARY_DIR / "subject_seed_summary.csv")
df_perm_subject = pd.read_csv(PERM_SUMMARY_DIR / "subject_summary_across_seeds.csv")
df_perm_model = pd.read_csv(PERM_SUMMARY_DIR / "model_summary_across_subjects.csv")

# Pre-stimulus-only control
prestim_subject_seed_path = PRESTIM_SUMMARY_DIR / "subject_seed_summary.csv"
prestim_subject_path = PRESTIM_SUMMARY_DIR / "subject_summary_across_seeds.csv"
prestim_model_path = PRESTIM_SUMMARY_DIR / "model_summary_across_subjects.csv"

df_prestim_subject_seed = (
    pd.read_csv(prestim_subject_seed_path)
    if prestim_subject_seed_path.exists()
    else None
)

df_prestim_subject = (
    pd.read_csv(prestim_subject_path)
    if prestim_subject_path.exists()
    else None
)

df_prestim_model = (
    pd.read_csv(prestim_model_path)
    if prestim_model_path.exists()
    else None
)

# LOSO
df_loso_subject_seed = pd.read_csv(LOSO_SUMMARY_DIR / "loso_subject_seed_summary.csv")
df_loso_subject = pd.read_csv(LOSO_SUMMARY_DIR / "loso_subject_summary_across_seeds.csv")
df_loso_model = pd.read_csv(LOSO_SUMMARY_DIR / "loso_model_summary_across_subjects.csv")

In [ ]:
print("Loaded tables:")
print("-" * 60)
print("Main subject summary     :", df_main_subject.shape)
print("Main model summary       :", df_main_model.shape)
print("Perm subject summary     :", df_perm_subject.shape)
print("Perm model summary       :", df_perm_model.shape)
print("Prestim subject summary  :", None if df_prestim_subject is None else df_prestim_subject.shape)
print("Prestim model summary    :", None if df_prestim_model is None else df_prestim_model.shape)
print("LOSO subject summary     :", df_loso_subject.shape)
print("LOSO model summary       :", df_loso_model.shape)

## 3. Inspect the main benchmark model ranking

This is the primary benchmark and therefore the main starting point for model selection.

In [ ]:
display(round_numeric_columns(df_main_model, decimals=4))

In [ ]:
print_model_ranking(df_main_model, metric_col="mean_subject_auc")

## 4. Inspect the control summaries

A model is more trustworthy if it performs well on the main benchmark **and** clearly weakens under the controls.

In [ ]:
print("Permuted-label control")
display(round_numeric_columns(df_perm_model, decimals=4))

if df_prestim_model is not None:
    print("Pre-stimulus-only control")
    display(round_numeric_columns(df_prestim_model, decimals=4))

## 5. Inspect the LOSO summary

LOSO is supporting evidence that helps quantify cross-subject generalization.

In [ ]:
display(round_numeric_columns(df_loso_model, decimals=4))

In [ ]:
print_model_ranking(df_loso_model, metric_col="mean_subject_auc")

## 6. Build direct comparison tables

We now compare the main benchmark against:

- permuted labels
- pre-stimulus-only
- LOSO

In [ ]:
df_main_vs_perm = make_main_vs_control_table(
    df_main_subject,
    df_perm_subject,
    value_col_main="auc_seed_mean",
    value_col_control="auc_seed_mean",
    main_label="main",
    control_label="permuted",
)

df_main_vs_loso = make_main_vs_control_table(
    df_main_subject,
    df_loso_subject,
    value_col_main="auc_seed_mean",
    value_col_control="auc_seed_mean",
    main_label="main",
    control_label="loso",
)

if df_prestim_subject is not None:
    df_main_vs_prestim = make_main_vs_control_table(
        df_main_subject,
        df_prestim_subject,
        value_col_main="auc_seed_mean",
        value_col_control="auc_seed_mean",
        main_label="main",
        control_label="prestim",
    )
else:
    df_main_vs_prestim = None

In [ ]:
df_main_vs_perm = round_numeric_columns(df_main_vs_perm, decimals=4)
df_main_vs_loso = round_numeric_columns(df_main_vs_loso, decimals=4)

print("Main vs permuted")
display(df_main_vs_perm)

save_dataframe(
    df_main_vs_perm,
    FINAL_TABLE_DIR / "main_vs_permuted_subject_table.csv",
)

if df_main_vs_prestim is not None:
    df_main_vs_prestim = round_numeric_columns(df_main_vs_prestim, decimals=4)

    print("Main vs pre-stimulus-only")
    display(df_main_vs_prestim)

    save_dataframe(
        df_main_vs_prestim,
        FINAL_TABLE_DIR / "main_vs_prestim_subject_table.csv",
    )

print("Main vs LOSO")
display(df_main_vs_loso)

save_dataframe(
    df_main_vs_loso,
    FINAL_TABLE_DIR / "main_vs_loso_subject_table.csv",
)

## 7. Build a compact decision table

This table combines the main signals needed for final selection.

### Included evidence

For each model we summarize:

- main within-subject mean subject ROC-AUC
- main within-subject worst-subject ROC-AUC
- permuted-label mean subject ROC-AUC
- pre-stimulus-only mean subject ROC-AUC (if available)
- LOSO mean subject ROC-AUC
- LOSO worst-subject ROC-AUC

In [ ]:
decision_df = (
    df_main_model[[
        "model",
        "mean_subject_auc",
        "std_subject_auc",
        "median_subject_auc",
        "worst_subject_auc",
        "best_subject_auc",
        "mean_subject_pr_auc",
        "mean_subject_bal_acc",
        "mean_subject_acc",
        "mean_subject_f1",
    ]]
    .rename(columns={
        "mean_subject_auc": "main_mean_auc",
        "std_subject_auc": "main_std_auc",
        "median_subject_auc": "main_median_auc",
        "worst_subject_auc": "main_worst_auc",
        "best_subject_auc": "main_best_auc",
        "mean_subject_pr_auc": "main_mean_pr_auc",
        "mean_subject_bal_acc": "main_mean_bal_acc",
        "mean_subject_acc": "main_mean_acc",
        "mean_subject_f1": "main_mean_f1",
    })
    .copy()
)

decision_df = decision_df.merge(
    df_perm_model[["model", "mean_subject_auc", "worst_subject_auc"]].rename(columns={
        "mean_subject_auc": "perm_mean_auc",
        "worst_subject_auc": "perm_worst_auc",
    }),
    on="model",
    how="left",
)

if df_prestim_model is not None:
    decision_df = decision_df.merge(
        df_prestim_model[["model", "mean_subject_auc", "worst_subject_auc"]].rename(columns={
            "mean_subject_auc": "prestim_mean_auc",
            "worst_subject_auc": "prestim_worst_auc",
        }),
        on="model",
        how="left",
    )

decision_df = decision_df.merge(
    df_loso_model[["model", "mean_subject_auc", "worst_subject_auc"]].rename(columns={
        "mean_subject_auc": "loso_mean_auc",
        "worst_subject_auc": "loso_worst_auc",
    }),
    on="model",
    how="left",
)

In [ ]:
decision_df["delta_main_minus_perm"] = (
        decision_df["main_mean_auc"] - decision_df["perm_mean_auc"]
)

if "prestim_mean_auc" in decision_df.columns:
    decision_df["delta_main_minus_prestim"] = (
            decision_df["main_mean_auc"] - decision_df["prestim_mean_auc"]
    )

decision_df["delta_main_minus_loso"] = (
        decision_df["main_mean_auc"] - decision_df["loso_mean_auc"]
)

decision_df = round_numeric_columns(
    decision_df.sort_values("main_mean_auc", ascending=False),
    decimals=4,
)

display(decision_df)

save_dataframe(
    decision_df,
    FINAL_TABLE_DIR / "decision_table.csv",
)

## 8. Define the final selection rule

The final selection rule used here is:

### Primary criterion
Choose the strongest model on the **main within-subject grouped benchmark**.

### Validity criterion
Prefer models whose performance clearly weakens under:
- permuted labels
- pre-stimulus-only

### Generalization criterion
Prefer models that remain competitive under LOSO and do not collapse completely on the worst subject.

### Practical criterion
If two models are very close, prefer the simpler and more stable one unless the more complex model provides a meaningful advantage.

This notebook will compute the evidence table, but the final written recommendation should still be interpreted carefully rather than treated as a blind automatic rule.

In [ ]:
ranking_cols = [
    "main_mean_auc",
    "main_worst_auc",
    "loso_mean_auc",
    "loso_worst_auc",
]

decision_ranked = decision_df.sort_values(
    by=ranking_cols,
    ascending=False,
).reset_index(drop=True)

decision_ranked = round_numeric_columns(decision_ranked, decimals=4)

display(decision_ranked)

save_dataframe(
    decision_ranked,
    FINAL_TABLE_DIR / "decision_ranked.csv",
)

## 9. Plot model rankings from the three main evidence sources

In [ ]:
fig_main_rank = plot_model_ranking_bars(
    df_main_model,
    metric_col="mean_subject_auc",
    title=f"Main within-subject benchmark: model ranking — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
    ylabel="Mean subject ROC-AUC",
)
fig_main_rank

In [ ]:
fig_perm_rank = plot_model_ranking_bars(
    df_perm_model,
    metric_col="mean_subject_auc",
    title=f"Permuted-label control: model ranking — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
    ylabel="Mean subject ROC-AUC",
)
fig_perm_rank

In [ ]:
if df_prestim_model is not None:
    fig_prestim_rank = plot_model_ranking_bars(
        df_prestim_model,
        metric_col="mean_subject_auc",
        title=f"Pre-stimulus-only control: model ranking — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
        ylabel="Mean subject ROC-AUC",
    )
    display(fig_prestim_rank)
else:
    fig_prestim_rank = None

In [ ]:
fig_loso_rank = plot_model_ranking_bars(
    df_loso_model,
    metric_col="mean_subject_auc",
    title=f"LOSO benchmark: model ranking — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
    ylabel="Mean subject ROC-AUC",
)
fig_loso_rank

## 10. Plot direct benchmark comparisons

In [ ]:
fig_main_vs_perm = plot_main_vs_control(
    df_main_vs_perm,
    main_col="main_value",
    control_col="permuted_value",
    title=f"Main benchmark vs permuted-label control — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
)
fig_main_vs_perm

In [ ]:
if df_main_vs_prestim is not None:
    fig_main_vs_prestim = plot_main_vs_control(
        df_main_vs_prestim,
        main_col="main_value",
        control_col="prestim_value",
        title=f"Main benchmark vs pre-stimulus-only control — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
    )
    display(fig_main_vs_prestim)
else:
    fig_main_vs_prestim = None

In [ ]:
fig_main_vs_loso = plot_main_vs_control(
    df_main_vs_loso,
    main_col="main_value",
    control_col="loso_value",
    title=f"Main benchmark vs LOSO — {cfg_main.dataset.dataset}, {cfg_main.dataset.task}",
)
fig_main_vs_loso

## 11. Final recommendation table

This is the compact table I would use as the main decision table in the thesis workflow.

In [ ]:
final_cols = [
    "model",
    "main_mean_auc",
    "main_worst_auc",
    "perm_mean_auc",
    "loso_mean_auc",
    "loso_worst_auc",
    "delta_main_minus_perm",
    "delta_main_minus_loso",
]

if "prestim_mean_auc" in decision_ranked.columns:
    final_cols.extend([
        "prestim_mean_auc",
        "delta_main_minus_prestim",
    ])

final_recommendation_table = round_numeric_columns(
    decision_ranked[final_cols],
    decimals=4,
)

display(final_recommendation_table)

save_dataframe(
    final_recommendation_table,
    FINAL_TABLE_DIR / "final_recommendation_table.csv",
)

## 12. Draft thesis-style interpretation

This section is intentionally written in a form that can be adapted into the thesis text.

In [ ]:
best_model = decision_ranked.iloc[0]["model"]

print(f"Top-ranked model by the current decision table: {best_model}")
print()
print(
    "Important: this is a dataset-level evidence summary. "
    "For the thesis-level final choice, compare this result with the other replication datasets "
    "and interpret the pattern across all datasets."
)

### Draft interpretation template

The final model-selection decision should be based primarily on the grouped within-subject benchmark, because this benchmark most directly reflects the intended subject-specific stimulus-classification setting of the thesis. However, strong performance on the main benchmark alone is not sufficient. A credible final choice should also exhibit a clear drop under the anti-overfitting controls and should remain reasonably competitive under cross-subject LOSO evaluation.

Accordingly, the final decision should be justified using three layers of evidence:

1. **Primary performance** on the main within-subject grouped benchmark
2. **Validity checks** using permuted-label and pre-stimulus-only controls
3. **Supporting generalization evidence** from LOSO evaluation

The model that should be selected is the one that provides the strongest overall balance across these dimensions rather than merely the highest isolated score.

## 13. Honest decision checklist

Before finalizing the model, review these questions carefully:

### A. Main benchmark
- Which model has the highest mean subject ROC-AUC?
- Which model has the strongest worst-subject ROC-AUC?
- Are differences large enough to matter, or are they small?

### B. Controls
- Does performance collapse under permuted labels?
- Does pre-stimulus-only performance remain clearly below the main benchmark?
- Does any model behave suspiciously well under a control?

### C. LOSO
- Does the model remain competitive on unseen subjects?
- Is the drop from within-subject to LOSO reasonable or extreme?
- Does the model collapse on weak subjects?

### D. Practicality
- Is the model simple to interpret and defend?
- Is the performance gain worth the added complexity?
- Is the model stable enough for thesis use?

If the answers are mixed, the thesis should say so directly.

## 14. Recommended selection wording options

These templates are intentionally cautious and should be filled using the actual observed results.

### Option 1 — EEGNet is clearly justified
EEGNet was selected as the final classification model because it provided the strongest overall performance on the primary within-subject grouped benchmark while remaining competitive under LOSO evaluation. In addition, its performance degraded appropriately under the permuted-label and pre-stimulus-only controls, supporting the interpretation that it learned meaningful post-stimulus structure rather than exploiting trivial or leaked information.

### Option 2 — Classical model remains preferable
Although EEGNet was competitive in some settings, the classical baseline ultimately provided the strongest overall balance of performance, robustness, and interpretability. In particular, the classical model remained highly competitive on the primary within-subject grouped benchmark and did not show a sufficiently large and consistent disadvantage relative to EEGNet to justify preferring the more complex neural model.

### Option 3 — Mixed conclusion
The experiments did not identify a single overwhelmingly dominant model across all evaluation settings. The final choice was therefore based on the primary within-subject grouped benchmark, with the anti-overfitting controls and LOSO evaluation used as supporting evidence. Under this criterion, the selected model provided the most defensible overall balance for the thesis objectives.

## 15. Save final decision artifacts

In [ ]:
save_figure(
    fig_main_rank,
    FINAL_FIG_DIR / "main_model_ranking.png",
)

save_figure(
    fig_perm_rank,
    FINAL_FIG_DIR / "permuted_model_ranking.png",
)

if fig_prestim_rank is not None:
    save_figure(
        fig_prestim_rank,
        FINAL_FIG_DIR / "prestim_model_ranking.png",
    )

save_figure(
    fig_loso_rank,
    FINAL_FIG_DIR / "loso_model_ranking.png",
)

save_figure(
    fig_main_vs_perm,
    FINAL_FIG_DIR / "main_vs_permuted.png",
)

if fig_main_vs_prestim is not None:
    save_figure(
        fig_main_vs_prestim,
        FINAL_FIG_DIR / "main_vs_prestim.png",
    )

save_figure(
    fig_main_vs_loso,
    FINAL_FIG_DIR / "main_vs_loso.png",
)

save_experiment_metadata(
    cfg_main,
    extra={
        "notebook": "05_final_model_selection.ipynb",
        "status": "completed",
        "experiment_key": EXPERIMENT_NAME,
        "dataset": cfg_main.dataset.dataset,
        "task": cfg_main.dataset.task,
        "top_ranked_model": str(best_model),
        "models_considered": sorted(decision_ranked["model"].unique().tolist()),
        "final_table_dir": str(FINAL_TABLE_DIR),
        "final_figure_dir": str(FINAL_FIG_DIR),
        "note": (
            "This is a dataset-level model-selection evidence summary. "
            "Final thesis-level interpretation should consider all replication datasets."
        ),
    },
)

print("Saved model-selection tables to:", FINAL_TABLE_DIR)
print("Saved model-selection figures to:", FINAL_FIG_DIR)

## 16. Final notebook conclusion

This notebook does not claim that the selected model is universally best in every setting.
Instead, it makes the final thesis decision using a layered and transparent logic:

- the **within-subject grouped benchmark** is the primary selection benchmark
- the **controls** test whether the benchmark is credible
- the **LOSO benchmark** measures supporting generalization behavior

The final selected model should therefore be the one that is **most defensible overall**, not merely the one with the highest isolated score.